# R9 follow-up testbed: soft \(v_1\) prior vs radial \(v_1\)-only trust region

**Purpose:** choose the next wake-rescue formulation using the real Re=100 CFD modes **before**
spending hours training ModalPINN again.

This notebook compares two candidates:

### Candidate A — annealed soft prior, only on downstream \(v_1\)

\[
L_K =
\frac{\sum_i W_i\,|\hat v_1(x_i,y_i)-S_{v1}(x_i,y_i)|^2}
     {\sum_i W_i\,|S_{v1}(x_i,y_i)|^2+\epsilon}.
\]

The mask \(W\) uses only geometry + the analytical prior amplitude, never CFD truth.

We test:

- how the result depends on where the prior starts downstream;
- whether the prior still pushes a collapsed wake away from zero;
- how much amplitude/phase bias it introduces;
- a full \((\alpha,\phi)\) landscape along the CFD wake direction.

### Candidate B — radial trust region, only on downstream \(v_1\)

\[
\hat v_1 =
S_{v1}
+
\rho |S_{v1}|
\frac{N}{\sqrt{1+|N|^2}}
\]

inside a prior-defined trusted wake region. Outside that region the network remains free.

We test:

- what fraction of the real CFD mode is reachable;
- how much CFD wake energy the trusted region covers;
- how much irreducible bias the trust region would impose;
- how wake-amplitude reachability changes from \(\alpha=0\) to \(\alpha=1.5\).

### Fairness rule

The **R9 street is still built from the 32 pressure taps + classical relations**.
The full CFD field is used only to evaluate candidate priors after they have been defined.

### Compute requirement

**CPU is enough. GPU is not used.** Expected runtime is roughly the same order as the previous
diagnostic notebook (usually a couple of minutes).

## 1. Mount Drive and locate the CFD + completed R9 prior

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DRIVE_DATA = '/content/drive/MyDrive/ModalPINN_data/fixed_cylinder_atRe100'
LOCAL_DATA = '/content/fixed_cylinder_atRe100'
RESULT_ROOT = '/content/drive/MyDrive/ModalPINN_results'
OUT_DIR = os.path.join(RESULT_ROOT, 'R9_v1_candidate_test')
os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(DRIVE_DATA), f'Dataset not found: {DRIVE_DATA}'

if not os.path.exists(LOCAL_DATA) or os.path.getsize(LOCAL_DATA) != os.path.getsize(DRIVE_DATA):
    print('Copying CFD data from Drive to local Colab storage...')
    shutil.copyfile(DRIVE_DATA, LOCAL_DATA)
else:
    print('Local CFD copy already exists.')

prior_candidates = sorted(
    glob.glob(os.path.join(RESULT_ROOT, 'R9_TRUST_street_*', 'street_prior_used.npz')),
    key=os.path.getmtime
)
if prior_candidates:
    PRIOR_PATH = prior_candidates[-1]
    print('Using completed-R9 prior:', PRIOR_PATH)
else:
    PRIOR_PATH = '/content/street_prior_Ntap32.npz'
    print('Saved R9 prior not found; fallback regeneration will be used.')

print('Outputs:', OUT_DIR)

## 2. Exact R9 data reader and prior generator

In [ ]:
%%writefile /content/text_flow.py
"""

Author: Mouad Boudina
From: https://zenodo.org/record/5039610


The flow file structure is the following:

Re Ur
(blank line)
Nt N_nodes (Nt = length of the timeline of the flow simulation)
(blank line)
t0
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...
t1
node0_x node0_y U(node0) V(node0) p(node0)
node1_x node1_y U(node1) V(node1) p(node1)
...

"""
import time
import numpy as np
#==============================================================================

def floatIt(l):
    return np.array([float(e) for e in l])

def intIt(l):
    return np.array([int(e) for e in l])

def read_flow(infile):
    f = open(infile, 'r')

    t1 = time.process_time()

    print('Reading flow...')

    Re, Ur = floatIt(f.readline().strip().split())

    f.readline() # blank line

    Nt, N_nodes = intIt(f.readline().strip().split())

    f.readline()

    times = []

    nodes_X, nodes_Y = [], []
    Us, Vs, ps = [], [], []

    for n in range(Nt):
        tn = float(f.readline().strip())
        times.append(tn)

        print('%.3f' % tn)

        tmp_nodes_X, tmp_nodes_Y = [], []
        tmp_Us, tmp_Vs, tmp_ps = [], [], []

        for k in range(N_nodes):
            x, y, U, V, p = floatIt(f.readline().strip().split())

            tmp_nodes_X.append(x)
            tmp_nodes_Y.append(y)

            tmp_Us.append(U)
            tmp_Vs.append(V)
            tmp_ps.append(p)

        nodes_X.append(tmp_nodes_X)
        nodes_Y.append(tmp_nodes_Y)

        Us.append(tmp_Us)
        Vs.append(tmp_Vs)
        ps.append(tmp_ps)

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

    return Re, Ur, np.array(times), \
           np.array(nodes_X), np.array(nodes_Y), \
           np.array(Us), np.array(Vs), np.array(ps)

def write_flow(flow, outfile):
    f = open(outfile, 'w')

    t1 = time.process_time()

    print('Writing flow...')

    f.write('%.0f %.1f\n' % (flow.Re, flow.Ur))
    f.write('\n') # blank line

    Nt, N_nodes = len(flow.times), len(flow.nodes_X[0])

    f.write('%d %d\n' % (Nt, N_nodes))
    f.write('\n')

    for n in range(Nt):
        tn = flow.times[n]

        print('%.6f' % tn)

        f.write('%.6f\n' % tn)

        for k in range(N_nodes):
            f.write('%13.9f %13.9f %13.9f %13.9f %13.9f\n' %\
                    (flow.nodes_X[n, k],
                     flow.nodes_Y[n, k],
                     flow.Us[n, k],
                     flow.Vs[n, k],
                     flow.ps[n, k]))

    cpu_time = time.process_time() - t1
    print('Done!')
    print('CPU_TIME = %f seconds' % cpu_time)

    f.close()

In [ ]:
%%writefile /content/street_prior.py
"""R9: derive the closed-form vortex-street prior from the 32 wall taps.

Standalone, numpy-only (no TF). Run BEFORE training:

    python street_prior.py --DataFile Data/fixed_cylinder_atRe100 --NTaps 32

Writes street_prior_Ntap<N>.npz with the closed-form street parameters that
ModalPINN_VortexShedding.py --TrustStreet consumes.

EVERY number here derives from the tap pressures + classical physics:
- omega0: nonlinear sinusoid fit to the tap-integrated lift series
- Gamma:  von Karman drag relation == tap-measured pressure drag / 0.75
          (0.75 = pressure share of total drag at Re~100, textbook value)
- Uc, a:  self-consistent street kinematics (Uc = 1 - Gamma/(sqrt8 a),
          a = 2 pi Uc / omega0)
- xf, r0, phase: matching the IMAGE-SYSTEM street's induced k=1 surface
          pressure pattern to the measured tap k=1 harmonics (the
          Milne-Thomson images make the surface pattern orientation-aware)
- closed-form calibration: the TF-portable single-harmonic expansion is
          aligned (phase offset + amplitude scale) against the numeric
          Lamb-Oseen street ON WAKE PROBE POINTS - a street-to-street
          calibration, no reference data involved.

The reference CFD file is read ONLY to extract the tap pressures - the
exact signals the training script itself trains on.

Method developed and validated in R9_wake_rescue/ (see REPORT.md there).
"""
import argparse
import os

import numpy as np
from scipy.optimize import least_squares

from text_flow import read_flow

# geometry, matching ModalPINN_VortexShedding.py
X_C, Y_C, R_C = 0.0, 0.0, 0.5
LXMIN, LXMAX, LYMIN, LYMAX = -4.0, 8.0, -4.0, 4.0
GEOM = [LXMIN, LXMAX, LYMIN, LYMAX, X_C, Y_C, R_C]
D = 2 * R_C
RE = 100.0
NU = 1.0 / RE
HA_RATIO = 0.281


# ===========================================================================
# numeric street (Lamb-Oseen rows + Milne-Thomson images + dipole)
# - reference implementation for the fit; identical math to
#   R9_wake_rescue/src/analytic_street.py
# ===========================================================================
class Street:
    def __init__(self, Gamma, U_c, x_f=1.0, r0=0.3, phase=0.0, omega=1.036,
                 ramp=0.75):
        self.G, self.Uc, self.omega = Gamma, U_c, omega
        self.a = 2 * np.pi * U_c / omega
        self.h = HA_RATIO * self.a
        self.xf, self.r0, self.phase, self.ramp = x_f, r0, phase, ramp

    def _vortex_positions(self, t, nwin=30):
        ks = np.arange(-nwin, nwin + 1)
        shift = (self.Uc * t + self.phase / self.omega * self.Uc) % self.a
        xu = self.xf + self.a * ks + shift
        xl = self.xf + self.a * (ks + 0.5) + shift
        return (np.stack([xu, np.full_like(xu, +self.h / 2)], 1),
                np.stack([xl, np.full_like(xl, -self.h / 2)], 1))

    def _induced(self, pts, vort_xy, gamma, core_from_x=None):
        dx = pts[:, None, 0] - vort_xy[None, :, 0]
        dy = pts[:, None, 1] - vort_xy[None, :, 1]
        r2 = dx ** 2 + dy ** 2 + 1e-12
        xv = vort_xy[:, 0] if core_from_x is None else core_from_x
        rc2 = self.r0 ** 2 + 4 * NU * np.clip(xv - self.xf, 0, None) / self.Uc
        fac = (1 - np.exp(-r2 / rc2[None, :])) / (2 * np.pi * r2)
        return (-gamma * dy * fac).sum(1), (gamma * dx * fac).sum(1)

    def velocity(self, pts, t):
        up, lo = self._vortex_positions(t)
        uu, vu = self._induced(pts, up, -self.G)
        ul, vl = self._induced(pts, lo, +self.G)
        u, v = uu + ul, vu + vl
        a2 = R_C ** 2
        for row, g in ((up, -self.G), (lo, +self.G)):
            r2v = row[:, 0] ** 2 + row[:, 1] ** 2
            img = row * (a2 / r2v)[:, None]
            ui, vi = self._induced(pts, img, -g, core_from_x=row[:, 0])
            u, v = u + ui, v + vi
        env = 0.5 * (1 + np.tanh((pts[:, 0] - self.xf) / self.ramp))
        x, y = pts[:, 0], pts[:, 1]
        r2 = x ** 2 + y ** 2 + 1e-12
        u_mean = 1.0 - a2 * (x ** 2 - y ** 2) / r2 ** 2
        v_mean = -a2 * 2 * x * y / r2 ** 2
        return u_mean + u * env, v_mean + v * env

    def pressure(self, pts, t):
        u, v = self.velocity(pts, t)
        return -0.5 * ((u - self.Uc) ** 2 + v ** 2)

    def modes(self, pts, nk=3, nt=16):
        T = 2 * np.pi / self.omega
        ts = np.arange(nt) * T / nt
        U = np.empty((nt, len(pts))); V = np.empty_like(U); P = np.empty_like(U)
        for i, t in enumerate(ts):
            U[i], V[i] = self.velocity(pts, t)
            P[i] = self.pressure(pts, t)
        out = {}
        for name, F in (('u', U), ('v', V), ('p', P)):
            c = np.fft.fft(F, axis=0) / nt
            out[name] = [c[0].real] + [c[k] for k in range(1, nk + 1)]
        return out


def uc_of_gamma(G, omega):
    Uc = 0.85
    for _ in range(50):
        a = 2 * np.pi * Uc / omega
        Uc_new = 1.0 - G / (np.sqrt(8.0) * a)
        if abs(Uc_new - Uc) < 1e-12:
            break
        Uc = Uc_new
    return Uc, 2 * np.pi * Uc / omega


def karman_drag_CD(G, Uc, a, h):
    u_ind = G / (np.sqrt(8.0) * a)
    return ((G * h / a) * (1.0 - 2.0 * u_ind) + G ** 2 / (2 * np.pi * a)) \
        / (0.5 * D)


# ===========================================================================
# closed-form street (TF-portable) - identical math to
# R9_wake_rescue/src/closed_form_street.py, numpy backend
# ===========================================================================
def cf_modes_uv(x, y, prm, nk=3):
    """One-sided modes k=1..nk of the closed-form street. Returns us, vs."""
    G, Uc, xf, r0, phase, omega, ramp, delta = (
        prm['Gamma'], prm['Uc'], prm['xf'], prm['r0'], prm['phase'],
        prm['omega'], prm.get('ramp', 0.75), prm.get('delta', 0.35))
    a = 2 * np.pi * Uc / omega
    h = HA_RATIO * a
    env = 0.5 * (1 + np.tanh((x - xf) / ramp))
    rc2 = r0 ** 2 + 4 * NU * np.clip(x - xf, 0, None) / Uc
    us, vs = [], []
    for k in range(1, nk + 1):
        att = np.exp(-(np.pi * k) ** 2 * rc2 / a ** 2)
        tot_u = np.zeros_like(x, dtype=complex)
        tot_v = np.zeros_like(x, dtype=complex)
        for y_row, sgn_row, x0 in ((+h / 2, -1.0, xf), (-h / 2, +1.0, xf + a / 2)):
            yp = y - y_row
            sabs = np.sqrt(yp ** 2 + delta ** 2) - delta
            sgn = -np.tanh(yp / delta)
            Dk = np.exp(-2 * np.pi * k * sabs / a)
            ph = -2 * np.pi * k * (x - x0) / a - k * phase
            Ek = np.cos(ph) + 1j * np.sin(ph)
            base = sgn_row * G / (2 * a) * Ek * Dk * att
            tot_u = tot_u + sgn * base
            tot_v = tot_v + 1j * base
        us.append(tot_u * env)
        vs.append(tot_v * env)
    return us, vs


# ===========================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--DataFile', default='Data/fixed_cylinder_atRe100')
    ap.add_argument('--NTaps', type=int, default=32)
    ap.add_argument('--Out', default=None)
    args = ap.parse_args()
    out_path = args.Out or f'street_prior_Ntap{args.NTaps}.npz'

    # ---- 1. tap pressures - same selection logic as Load_train_data_desync
    # cut_simu_cylinder_only (transcribed, not imported: that module imports
    # tensorflow, which this numpy-only script must not depend on).
    Re_, Ur_, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(args.DataFile)
    eps = 1e-5
    r_all = np.sqrt((nodes_X[0, :] - X_C) ** 2 + (nodes_Y[0, :] - Y_C) ** 2)
    idx_cyl = np.argwhere((r_all - R_C) ** 2 < eps)[:, 0]
    xc_all, yc_all = nodes_X[0, idx_cyl], nodes_Y[0, idx_cyl]
    s_lin = np.linspace(0., 1., args.NTaps, endpoint=False)
    x_t = X_C + R_C * np.cos(2 * np.pi * s_lin)
    y_t = Y_C + R_C * np.sin(2 * np.pi * s_lin)
    pick = np.array([np.argmin((xc_all - x_t[k]) ** 2 + (yc_all - y_t[k]) ** 2)
                     for k in range(args.NTaps)])
    print('Cylinder taps requested: %d, distinct mesh nodes found: %d'
          % (args.NTaps, len(np.unique(pick))))
    x_cyl, y_cyl = xc_all[pick], yc_all[pick]
    p_cyl = Ps[:, idx_cyl[pick]]             # (Nt, NTaps)
    t = np.asarray(times) - times[0]
    print(f'taps: {p_cyl.shape}, t in [0, {t[-1]:.1f}]')

    theta = np.arctan2(y_cyl - Y_C, x_cyl - X_C)
    order = np.argsort(theta)
    th_s, p_s = theta[order], p_cyl[:, order]
    dth = np.diff(np.concatenate([th_s, [th_s[0] + 2 * np.pi]]))
    w = 0.5 * (dth + np.roll(dth, 1))
    CD = -(p_s * np.cos(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)
    CL = -(p_s * np.sin(th_s)[None, :] * w[None, :]).sum(1) * R_C / (0.5 * D)

    # ---- 2. omega0 from a nonlinear sinusoid fit to CL
    z = CL - CL.mean()
    crossings = np.where(np.diff(np.sign(z)) != 0)[0]
    w_init = np.pi / np.mean(np.diff(t[crossings]))
    fit = least_squares(
        lambda prm: prm[0] * np.sin(prm[2] * t + prm[1]) + prm[3] - CL,
        [0.5 * (CL.max() - CL.min()), 0.0, w_init, CL.mean()], method='lm')
    omega = abs(float(fit.x[2]))
    CD0 = float(CD.mean())
    print(f'omega0_hat = {omega:.5f}  CD_pressure = {CD0:.4f}')

    # ---- 3. per-tap k=1 harmonics
    cols = [np.ones_like(t), np.cos(omega * t), np.sin(omega * t)]
    A = np.stack(cols, 1)
    cf_, *_ = np.linalg.lstsq(A, p_s, rcond=None)
    p1_meas = 0.5 * (cf_[1] - 1j * cf_[2])

    # ---- 4. Gamma from the Karman drag relation (bisection)
    CD_target = CD0 / 0.75
    lo, hi = 0.5, 6.0
    for _ in range(60):
        G = 0.5 * (lo + hi)
        Uc, a = uc_of_gamma(G, omega)
        if karman_drag_CD(G, Uc, a, HA_RATIO * a) < CD_target:
            lo = G
        else:
            hi = G
    G = 0.5 * (lo + hi)
    Uc, a = uc_of_gamma(G, omega)
    print(f'Gamma = {G:.3f}  Uc = {Uc:.3f}  a = {a:.3f}')

    # ---- 5. xf, r0, phase from the tap k=1 pattern (image street)
    tap_pts = np.stack([R_C * np.cos(th_s), R_C * np.sin(th_s)], 1)
    best = None
    for xf in (0.6, 0.8, 1.0, 1.2):
        for r0 in (0.2, 0.3, 0.4):
            st = Street(G, Uc, x_f=xf, r0=r0, omega=omega)
            sm = st.modes(tap_pts, nk=1, nt=16)
            p1s = sm['p'][1]
            corr = np.abs(np.vdot(p1s, p1_meas)) / (
                np.linalg.norm(p1s) * np.linalg.norm(p1_meas))
            phi = np.angle(np.vdot(p1s, p1_meas))
            if best is None or corr > best[0]:
                best = (corr, xf, r0, phi)
    corr_tap, xf, r0, phi = best
    print(f'xf = {xf}  r0 = {r0}  phase = {phi:+.3f}  tap-p1 corr = {corr_tap:.3f}')

    # ---- 6. calibrate the closed form against the numeric street
    num = Street(G, Uc, x_f=xf, r0=r0, phase=phi, omega=omega)
    rng = np.random.default_rng(3)
    pts = rng.uniform([1.0, -2.0], [8.0, 2.0], size=(1500, 2))
    sm = num.modes(pts, nk=3, nt=16)
    prm = dict(Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega, phase=phi)
    best = None
    for extra in np.linspace(-np.pi, np.pi, 48, endpoint=False):
        prm['phase'] = phi + extra
        us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
        inner = np.vdot(vs[0], sm['v'][1])
        corr = abs(inner) / (np.linalg.norm(vs[0])
                             * np.linalg.norm(sm['v'][1]) + 1e-30)
        score = corr - abs(np.angle(inner)) * 0.05
        if best is None or score > best[0]:
            best = (score, corr, extra)
    _, corr_cf, extra = best
    prm['phase'] = phi + extra
    us, vs = cf_modes_uv(pts[:, 0], pts[:, 1], prm, nk=1)
    amp_scale = float(np.linalg.norm(sm['v'][1]) / np.linalg.norm(vs[0]))
    # pressure anchor: p_k ~ -(1-Uc) u_k, amplitude-matched at k=1
    p1_approx = -(1.0 - Uc) * us[0] * amp_scale
    scale_p = float(np.linalg.norm(sm['p'][1]) / np.linalg.norm(p1_approx))
    print(f'closed-form calibration: corr vs numeric = {corr_cf:.3f}, '
          f'amp_scale = {amp_scale:.3f}, scale_p = {scale_p:.3f}')
    assert corr_cf > 0.95, 'closed-form street failed to match numeric street'

    np.savez(out_path,
             Gamma=G, Uc=Uc, xf=xf, r0=r0, omega=omega,
             phase=prm['phase'], amp_scale=amp_scale, scale_p=scale_p,
             ramp=0.75, delta=0.35,
             CD_pressure=CD0, tap_p1_corr=corr_tap,
             cf_corr_vs_numeric=corr_cf)
    print('saved', out_path)


if __name__ == '__main__':
    main()

## 3. Load or regenerate the exact completed-R9 pressure-derived prior

In [ ]:
if not os.path.exists(PRIOR_PATH):
    import subprocess
    print('Regenerating R9 prior from the 32 pressure taps...')
    subprocess.run([
        'python', '/content/street_prior.py',
        '--DataFile', LOCAL_DATA,
        '--NTaps', '32',
        '--Out', '/content/street_prior_Ntap32.npz'
    ], check=True)
    PRIOR_PATH = '/content/street_prior_Ntap32.npz'

sp_npz = np.load(PRIOR_PATH)
sp = {k: float(sp_npz[k]) for k in sp_npz.files}

print('Prior path:', PRIOR_PATH)
for k in ['omega','Gamma','Uc','xf','r0','phase','amp_scale',
          'scale_p','tap_p1_corr','cf_corr_vs_numeric']:
    if k in sp:
        print(f'{k:20s}: {sp[k]:.8g}')

## 4. Extract the real CFD \(k=1\) mode

Same Fourier convention as the previous diagnostic:

\[
q(t)=q_0+2\Re[\hat q_1e^{i\omega t}+\hat q_2e^{2i\omega t}+\cdots].
\]

In [ ]:
import sys
sys.path.insert(0, '/content')
from text_flow import read_flow

Re, Ur, times, nodes_X, nodes_Y, Us, Vs, Ps = read_flow(LOCAL_DATA)
times = np.asarray(times)
x_all = nodes_X[0]
y_all = nodes_Y[0]

in_box = ((x_all > -4.0) & (x_all < 8.0) &
          (y_all > -4.0) & (y_all < 4.0))
idx = np.where(in_box)[0]

x = x_all[idx].astype(float)
y = y_all[idx].astype(float)
U = Us[:, idx]
V = Vs[:, idx]

omega = sp['omega']
t = times-times[0]

cols = [np.ones_like(t)]
for k in range(1,4):
    cols += [np.cos(k*omega*t), np.sin(k*omega*t)]
B = np.stack(cols, axis=1)

def fit_modes(F):
    coef, *_ = np.linalg.lstsq(B, F, rcond=None)
    modes = [coef[0].astype(complex)]
    j = 1
    for k in range(1,4):
        modes.append(0.5*(coef[j]-1j*coef[j+1]))
        j += 2
    return modes

u_modes = fit_modes(U)
v_modes = fit_modes(V)
u1 = u_modes[1]
v1 = v_modes[1]

del U, V, Us, Vs, Ps, nodes_X, nodes_Y

r = np.sqrt(x*x+y*y)
base_wake = (x >= 0.5) & (x <= 8.0) & (np.abs(y) <= 2.5) & (r > 0.60)
far_eval = (x >= 3.0) & (x <= 8.0) & (np.abs(y) <= 2.0) & (r > 0.60)

print('Re =', Re)
print('omega =', omega)
print('cropped nodes =', len(x))
print('base wake nodes =', base_wake.sum())
print('far evaluation nodes =', far_eval.sum())

## 5. Reconstruct the exact R9 closed-form \(v_1\) prior

In [ ]:
HA_RATIO = 0.281
NU = 1.0/100.0

def cf_modes_uv_np(x, y, prm, nk=1):
    G, Uc, xf, r0, phase, omega = (
        prm['Gamma'], prm['Uc'], prm['xf'], prm['r0'],
        prm['phase'], prm['omega'])
    ramp = prm.get('ramp', 0.75)
    delta = prm.get('delta', 0.35)

    a = 2*np.pi*Uc/omega
    h = HA_RATIO*a
    env = 0.5*(1 + np.tanh((x-xf)/ramp))
    rc2 = r0**2 + 4*NU*np.clip(x-xf,0,None)/Uc

    us, vs = [], []
    for k in range(1,nk+1):
        att = np.exp(-(np.pi*k)**2*rc2/a**2)
        tot_u = np.zeros_like(x,dtype=complex)
        tot_v = np.zeros_like(x,dtype=complex)

        for y_row, sgn_row, x0 in (
            (+h/2, -1.0, xf),
            (-h/2, +1.0, xf+a/2)
        ):
            yp = y-y_row
            sabs = np.sqrt(yp**2+delta**2)-delta
            sgn = -np.tanh(yp/delta)
            Dk = np.exp(-2*np.pi*k*sabs/a)
            ph = -2*np.pi*k*(x-x0)/a-k*phase
            Ek = np.cos(ph)+1j*np.sin(ph)
            base = sgn_row*G/(2*a)*Ek*Dk*att
            tot_u += sgn*base
            tot_v += 1j*base

        us.append(tot_u*env)
        vs.append(tot_v*env)

    return us,vs

_, vs_cf = cf_modes_uv_np(x,y,sp,nk=1)
S = sp['amp_scale']*vs_cf[0]   # conventional CFD Fourier convention
T = v1                         # CFD truth, evaluation only

raw_corr = abs(np.vdot(S[far_eval],T[far_eval])) / (
    np.linalg.norm(S[far_eval])*np.linalg.norm(T[far_eval])+1e-30
)
raw_err = np.linalg.norm(S[far_eval]-T[far_eval]) / (
    np.linalg.norm(T[far_eval])+1e-30
)

print('Sanity check, far wake:')
print('  raw v1 correlation = %.6f' % raw_corr)
print('  raw v1 rel L2      = %.6f' % raw_err)
print('Expected from previous diagnostic: corr ~0.977, rel L2 ~0.284')

# Candidate A — annealed soft \(v_1\) prior

## 6. Define a fair prior mask

The **training mask must not use CFD truth**. We therefore build it only from:

- downstream coordinate \(x\);
- transverse coordinate \(y\);
- the analytical prior amplitude \(|S_{v1}|\).

For each start position \(x_s\) and amplitude gate \(\eta\):

\[
W = W_x W_y W_S.
\]

The CFD mode is used only to score how good that mask turns out to be.

In [ ]:
def soft_mask(xstart, eta, xwidth=0.30, ymax=2.0, ywidth=0.20):
    wx = 0.5*(1+np.tanh((x-xstart)/xwidth))
    wy = 0.5*(1-np.tanh((np.abs(y)-ymax)/ywidth))

    # Prior-amplitude gate. Scale is computed only from the prior downstream.
    geom = (x >= xstart) & (np.abs(y) <= ymax)
    smax = np.max(np.abs(S[geom])) if np.any(geom) else np.max(np.abs(S))
    threshold = eta*smax
    ws = (np.abs(S)**2) / (np.abs(S)**2 + threshold**2 + 1e-30)
    return wx*wy*ws

def weighted_metrics(Sref, Truth, W):
    den_s = np.sum(W*np.abs(Sref)**2)+1e-30
    den_t = np.sum(W*np.abs(Truth)**2)+1e-30

    err = np.sqrt(np.sum(W*np.abs(Sref-Truth)**2)/den_t)
    inner = np.sum(W*np.conj(Sref)*Truth)
    corr = abs(inner)/np.sqrt(den_s*den_t)

    # Along v(alpha)=alpha*T with real alpha.
    alpha_pref = np.real(np.sum(W*np.conj(Truth)*Sref))/den_t
    dL0 = -2*np.real(np.sum(W*np.conj(Truth)*Sref))/den_s

    # If amplitude AND global phase were free along the CFD shape.
    z_pref = np.sum(W*np.conj(Truth)*Sref)/den_t

    # Oracle complex scale of the prior itself.
    c_or = np.sum(W*np.conj(Sref)*Truth)/den_s
    err_or = np.sqrt(np.sum(W*np.abs(c_or*Sref-Truth)**2)/den_t)

    return {
        'rel_L2': err,
        'corr': corr,
        'MAC': corr**2,
        'alpha_pref_real_path': alpha_pref,
        'dL_dalpha_at_zero': dL0,
        'truth_shape_complex_gain_abs': abs(z_pref),
        'truth_shape_complex_phase_deg': np.degrees(np.angle(z_pref)),
        'oracle_prior_gain_abs': abs(c_or),
        'oracle_prior_phase_deg': np.degrees(np.angle(c_or)),
        'oracle_rel_L2': err_or,
        'weight_sum': np.sum(W),
    }

xstarts = [1.0,1.5,2.0,2.5,3.0,3.5]
etas = [0.00,0.03,0.05,0.10,0.15,0.20]

soft_rows = []
for xs in xstarts:
    for eta in etas:
        W = soft_mask(xs,eta)
        m = weighted_metrics(S,T,W)
        soft_rows.append({'xstart':xs,'eta':eta,**m})

soft = pd.DataFrame(soft_rows)

# A simple diagnostic score: high corr, preferred amplitude close to 1,
# and nonzero negative collapse gradient.
soft['score'] = (
    soft['corr']
    - 0.35*np.abs(soft['alpha_pref_real_path']-1.0)
    - 0.10*soft['rel_L2']
)
soft = soft.sort_values('score',ascending=False).reset_index(drop=True)

display(soft.head(18).round(4))
soft.to_csv(os.path.join(OUT_DIR,'soft_v1_mask_sweep.csv'),index=False)

best_soft = soft.iloc[0]
print('\nBest soft-mask diagnostic candidate:')
print(best_soft.round(5).to_string())

## 7. Soft-prior mask sweep plots

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
for eta in etas:
    sub = soft[soft.eta==eta].sort_values('xstart')
    ax.plot(sub.xstart,sub['corr'],'o-',label=f'eta={eta:g}')
ax.set_xlabel('prior start x/D')
ax.set_ylabel('weighted complex correlation')
ax.set_title('Soft v1 prior: correlation vs downstream start')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_corr_vs_xstart.png'),dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
for eta in etas:
    sub = soft[soft.eta==eta].sort_values('xstart')
    ax.plot(sub.xstart,sub['alpha_pref_real_path'],'o-',label=f'eta={eta:g}')
ax.axhline(1.0,linestyle='--')
ax.set_xlabel('prior start x/D')
ax.set_ylabel('preferred alpha along CFD wake direction')
ax.set_title('Soft v1 prior: amplitude bias')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_alpha_pref_vs_xstart.png'),dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
for eta in etas:
    sub = soft[soft.eta==eta].sort_values('xstart')
    ax.plot(sub.xstart,sub['dL_dalpha_at_zero'],'o-',label=f'eta={eta:g}')
ax.axhline(0.0,linestyle='--')
ax.set_xlabel('prior start x/D')
ax.set_ylabel('dL_K/dalpha at alpha=0')
ax.set_title('Soft v1 prior: collapse-rescue gradient')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_collapse_gradient.png'),dpi=180)
plt.show()

## 8. Full soft-prior amplitude/phase landscape for the best mask

We constrain the test path to

\[
v_1(\alpha,\phi)=\alpha e^{i\phi}v_1^{CFD}.
\]

The ideal minimum would sit near

\[
(\alpha,\phi)=(1,0).
\]

This tells us exactly how much the analytical prior tries to bias the real CFD mode before
annealing is switched off.

In [ ]:
xs_best = float(best_soft.xstart)
eta_best = float(best_soft.eta)
W_best = soft_mask(xs_best,eta_best)

alpha_grid = np.linspace(0,1.5,121)
phase_deg_grid = np.linspace(-45,45,121)
phase_grid = np.radians(phase_deg_grid)

den = np.sum(W_best*np.abs(S)**2)+1e-30
L = np.empty((len(phase_grid),len(alpha_grid)))

for i,ph in enumerate(phase_grid):
    for j,a in enumerate(alpha_grid):
        Q = a*np.exp(1j*ph)*T
        L[i,j] = np.sum(W_best*np.abs(Q-S)**2)/den

ij = np.unravel_index(np.argmin(L),L.shape)
amin = alpha_grid[ij[1]]
pmin = phase_deg_grid[ij[0]]

print('Soft-landscape minimum:')
print('  alpha = %.4f' % amin)
print('  phase = %.2f deg' % pmin)

fig, ax = plt.subplots(figsize=(8,5))
cs = ax.contourf(alpha_grid,phase_deg_grid,L,levels=40)
ax.plot([1],[0],'x',markersize=10,label='CFD truth')
ax.plot([amin],[pmin],'o',label='prior-loss minimum')
ax.set_xlabel('alpha')
ax.set_ylabel('global phase shift [deg]')
ax.set_title(f'Soft v1 prior landscape: xstart={xs_best:g}, eta={eta_best:g}')
ax.legend()
fig.colorbar(cs,ax=ax,label='normalized prior loss')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_alpha_phase_landscape.png'),dpi=180)
plt.show()

## 9. Annealing schedule — what we can and cannot decide without training

This notebook can verify that the prior has the **right direction** at the collapsed wake.

It cannot determine the perfect numerical \(\lambda_0\) relative to the full ModalPINN
physics/data losses without actually building that training graph.

So we only test the schedule shape here. A later short smoke-training run can screen
\(\lambda_0\) cheaply.

In [ ]:
s = np.linspace(0,1,501)

def cosine_to_zero(s,end=0.30):
    z = np.zeros_like(s)
    inside = s < end
    z[inside] = 0.5*(1+np.cos(np.pi*s[inside]/end))
    return z

lam_shape = cosine_to_zero(s,0.30)
dL0_best = float(best_soft['dL_dalpha_at_zero'])

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(s,lam_shape)
ax.set_xlabel('fraction of training completed')
ax.set_ylabel('lambda_K / lambda_0')
ax.set_title('Suggested soft-prior annealing shape')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_annealing_schedule.png'),dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(s,lam_shape*dL0_best)
ax.axhline(0,linestyle='--')
ax.set_xlabel('fraction of training completed')
ax.set_ylabel('(lambda_K/lambda_0) * dL/dalpha at collapse')
ax.set_title('Collapse-rescue drive disappears completely by 30% training')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'soft_annealed_collapse_drive.png'),dpi=180)
plt.show()

# Candidate B — radial \(v_1\)-only trust region

## 10. Define prior-only trusted regions

We do **not** trust the analytical street everywhere.

A point is included only if:

- \(x \ge x_s\);
- \(|y|\le2D\);
- \(|S_{v1}|\) exceeds a fraction \(\eta\) of the prior's maximum amplitude downstream.

All three conditions can be computed without CFD truth.

Within that trusted region:

\[
|v_1-S_{v1}| \le \rho |S_{v1}|.
\]

For any \(\rho<1\), exact \(v_1=0\) is impossible wherever \(S_{v1}\neq0\).

In [ ]:
def trust_mask(xstart,eta,ymax=2.0):
    geom = (x>=xstart) & (x<=8.0) & (np.abs(y)<=ymax) & (r>0.60)
    if not np.any(geom):
        return geom
    smax = np.max(np.abs(S[geom]))
    return geom & (np.abs(S) >= eta*smax)

def radial_stats(mask,rho):
    Sm = S[mask]
    Tm = T[mask]

    radius = rho*np.abs(Sm)
    dist_truth = np.abs(Tm-Sm)

    reachable = dist_truth <= radius + 1e-14
    outside = np.maximum(dist_truth-radius,0.0)

    # Relative irreducible error if each point is projected to the nearest
    # point in the allowed disk.
    bias_rel = np.linalg.norm(outside)/(np.linalg.norm(Tm)+1e-30)

    corr = abs(np.vdot(Sm,Tm))/(
        np.linalg.norm(Sm)*np.linalg.norm(Tm)+1e-30
    )

    # Wake-energy coverage relative to a fixed evaluation wake.
    energy_total = np.sum(np.abs(T[far_eval])**2)+1e-30
    coverage = np.sum(np.abs(Tm)**2)/energy_total

    floor_ratio = np.mean((1-rho)*np.abs(Sm))/(
        np.mean(np.abs(Tm))+1e-30
    )

    return {
        'truth_reachable_frac': np.mean(reachable),
        'irreducible_bias_rel_L2': bias_rel,
        'prior_corr_in_trust_region': corr,
        'CFD_farwake_energy_coverage': coverage,
        'mean_floor_over_mean_truth': floor_ratio,
        'n_trusted': int(mask.sum()),
        'zero_reachable_frac': 0.0 if rho < 1.0 else 1.0,
    }

rho_grid = np.arange(0.40,0.981,0.02)
xstart_grid = [1.5,2.0,2.5,3.0,3.5]
eta_grid = [0.03,0.05,0.10,0.15,0.20]

radial_rows = []

for xs in xstart_grid:
    for eta in eta_grid:
        M = trust_mask(xs,eta)
        if M.sum() < 10:
            continue
        for rho_ in rho_grid:
            st = radial_stats(M,rho_)
            radial_rows.append({
                'xstart':xs,
                'eta':eta,
                'rho':float(rho_),
                **st
            })

radial = pd.DataFrame(radial_rows)
radial.to_csv(os.path.join(OUT_DIR,'radial_v1_parameter_sweep.csv'),index=False)

# Preferred candidates: block zero, contain >=99% of CFD points in the trusted region,
# cover >=80% of the fixed far-wake CFD energy, and minimize rho (strongest floor)
# then minimize irreducible bias.
eligible = radial[
    (radial.truth_reachable_frac >= 0.99) &
    (radial.CFD_farwake_energy_coverage >= 0.80) &
    (radial.zero_reachable_frac == 0)
].copy()

if len(eligible):
    eligible = eligible.sort_values(
        ['rho','irreducible_bias_rel_L2','CFD_farwake_energy_coverage'],
        ascending=[True,True,False]
    )
    best_radial = eligible.iloc[0]
    print('Best radial candidate under >=99% truth reach + >=80% far-wake energy coverage:')
    print(best_radial.round(5).to_string())
else:
    best_radial = radial.sort_values(
        ['truth_reachable_frac','irreducible_bias_rel_L2'],
        ascending=[False,True]
    ).iloc[0]
    print('No candidate met both thresholds. Best available:')
    print(best_radial.round(5).to_string())

display(
    radial.sort_values(
        ['truth_reachable_frac','CFD_farwake_energy_coverage','rho'],
        ascending=[False,False,True]
    ).head(20).round(4)
)

## 11. Radial-trust sweep plots

In [ ]:
xs_r = float(best_radial.xstart)
eta_r = float(best_radial.eta)
sub = radial[(radial.xstart==xs_r)&(radial.eta==eta_r)].sort_values('rho')

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(sub.rho,sub.truth_reachable_frac,'o-')
ax.axhline(0.99,linestyle='--')
ax.set_xlabel('rho')
ax.set_ylabel('CFD truth reachable fraction')
ax.set_title(f'Radial v1 trust: xstart={xs_r:g}, eta={eta_r:g}')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'radial_truth_reach_vs_rho.png'),dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(sub.rho,sub.irreducible_bias_rel_L2,'o-')
ax.set_xlabel('rho')
ax.set_ylabel('minimum imposed relative L2 bias')
ax.set_title(f'Radial v1 trust: unavoidable bias vs rho')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'radial_bias_vs_rho.png'),dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(sub.rho,sub.mean_floor_over_mean_truth,'o-')
ax.set_xlabel('rho')
ax.set_ylabel('mean hard floor / mean CFD |v1|')
ax.set_title('Radial v1 trust: strength of the anti-collapse floor')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'radial_floor_vs_rho.png'),dpi=180)
plt.show()

## 12. Radial trust: global wake-amplitude feasibility

Along

\[
v_1(\alpha)=\alpha v_1^{CFD},
\]

we ask what fraction of trusted points can be represented by the radial trust region.

This directly checks the two endpoints we care about:

- \(\alpha=0\): dead wake should be unavailable;
- \(\alpha=1\): true CFD wake should be almost completely available.

In [ ]:
Mbest = trust_mask(float(best_radial.xstart),float(best_radial.eta))
rho_best = float(best_radial.rho)

Sm = S[Mbest]
Tm = T[Mbest]
radius = rho_best*np.abs(Sm)

alpha = np.linspace(0,1.5,301)
feasible_frac = []

for a in alpha:
    feasible = np.abs(a*Tm-Sm) <= radius + 1e-14
    feasible_frac.append(np.mean(feasible))

feasible_frac = np.asarray(feasible_frac)

print('At alpha=0: feasible fraction = %.6f' % feasible_frac[0])
print('At alpha=1: feasible fraction = %.6f' %
      feasible_frac[np.argmin(np.abs(alpha-1.0))])

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(alpha,feasible_frac)
ax.axvline(0,linestyle='--')
ax.axvline(1,linestyle='--')
ax.axhline(0.99,linestyle=':')
ax.set_xlabel('wake amplitude alpha')
ax.set_ylabel('fraction of trusted points reachable')
ax.set_title(
    f'Radial v1 feasibility: rho={rho_best:.2f}, '
    f'xstart={float(best_radial.xstart):g}, eta={float(best_radial.eta):g}'
)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR,'radial_alpha_feasibility.png'),dpi=180)
plt.show()

# Direct comparison

## 13. Compare what each candidate guarantees

This is a diagnostic comparison, not a final training result.

### Soft prior

**Advantage**
- does not permanently restrict the final solution;
- can be annealed exactly to zero;
- gives a measurable non-zero collapse gradient.

**Unknown until a short training smoke test**
- what \(\lambda_0\) is large enough to beat the pressure-only collapsed basin.

### Radial trust

**Advantage**
- mathematically removes exact collapse inside the trusted \(v_1\) region;
- no loss-weight tuning is needed for that guarantee.

**Cost**
- permanently restricts \(v_1\) in that region;
- therefore we must verify that CFD truth lies inside the allowed disks.

The next actual network experiment should be chosen from the numbers below.

In [ ]:
soft_out = {
    'xstart': float(best_soft.xstart),
    'eta': float(best_soft.eta),
    'corr': float(best_soft['corr']),
    'rel_L2': float(best_soft['rel_L2']),
    'alpha_pref': float(best_soft['alpha_pref_real_path']),
    'dL_dalpha_at_zero': float(best_soft['dL_dalpha_at_zero']),
    'oracle_rel_L2': float(best_soft['oracle_rel_L2']),
    'landscape_min_alpha': float(amin),
    'landscape_min_phase_deg': float(pmin),
}

radial_out = {
    'xstart': float(best_radial.xstart),
    'eta': float(best_radial.eta),
    'rho': float(best_radial.rho),
    'truth_reachable_frac': float(best_radial.truth_reachable_frac),
    'irreducible_bias_rel_L2': float(best_radial.irreducible_bias_rel_L2),
    'CFD_farwake_energy_coverage': float(best_radial.CFD_farwake_energy_coverage),
    'mean_floor_over_mean_truth': float(best_radial.mean_floor_over_mean_truth),
    'zero_reachable_frac': float(best_radial.zero_reachable_frac),
    'prior_corr_in_trust_region': float(best_radial.prior_corr_in_trust_region),
}

comparison = pd.DataFrame([
    {
        'candidate':'soft_annealed_v1',
        'xstart':soft_out['xstart'],
        'eta':soft_out['eta'],
        'rho':np.nan,
        'prior_corr':soft_out['corr'],
        'truth_reachable_frac':1.0,
        'hard_zero_blocked':False,
        'collapse_gradient':soft_out['dL_dalpha_at_zero'],
        'permanent_constraint':False,
    },
    {
        'candidate':'radial_v1_trust',
        'xstart':radial_out['xstart'],
        'eta':radial_out['eta'],
        'rho':radial_out['rho'],
        'prior_corr':radial_out['prior_corr_in_trust_region'],
        'truth_reachable_frac':radial_out['truth_reachable_frac'],
        'hard_zero_blocked':radial_out['zero_reachable_frac']==0.0,
        'collapse_gradient':np.nan,
        'permanent_constraint':True,
    }
])

display(comparison.round(5))
comparison.to_csv(os.path.join(OUT_DIR,'candidate_comparison.csv'),index=False)

result = {
    'soft_candidate':soft_out,
    'radial_candidate':radial_out,
    'notes':{
        'gpu_needed_for_this_notebook':False,
        'CFD_used_only_for_evaluation':True,
        'recommended_next_step':
            'Run a short ModalPINN smoke test for BOTH candidates before any full 9-hour production run.'
    }
}

with open(os.path.join(OUT_DIR,'candidate_verdict.json'),'w') as f:
    json.dump(result,f,indent=2)

print(json.dumps(result,indent=2))
print('\nSaved:',os.path.join(OUT_DIR,'candidate_verdict.json'))

## 14. What to send back

After **Run all**, send me:

1. `candidate_verdict.json`
2. `soft_v1_mask_sweep.csv`
3. `radial_v1_parameter_sweep.csv`

or simply zip the whole:

`MyDrive/ModalPINN_results/R9_v1_candidate_test/`

Then we can choose the exact formulation for a **short smoke-training run**.

**Do not launch another long production training yet.**